# P4 · GNN — Drug Candidate Ranking
**Input:** `results/tables/dgi_edges_gnn.csv`  
**Outputs:** `models/gcn_best.pt` · `results/tables/gnn_drug_ranking.csv`
· `results/tables/gnn_node_embeddings.csv` · `results/figures/gnn_*.png`

---

### Running on Google Colab with GPU

1. Upload `dgi_edges_gnn.csv` to the Colab session (Files → Upload)
2. Set `COLAB_MODE = True` in the cell below
3. Runtime → Change runtime type → **T4 GPU** → Run all

When running locally, keep `COLAB_MODE = False` (auto-detected).


## Environment detection

In [ ]:
import sys, os
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

COLAB_MODE = IN_COLAB
print(f"Running on {'Google Colab' if IN_COLAB else 'local machine'}")

## Install dependencies (Colab only)

In [ ]:
if COLAB_MODE:
    import subprocess
    import torch as _torch
    torch_ver = _torch.__version__.split("+")[0]
    cuda_tag  = ("cu" + _torch.version.cuda.replace(".", "")
                 if _torch.cuda.is_available() else "cpu")
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "torch-geometric","torch-scatter","torch-sparse","torch-cluster",
                    "--find-links",
                    f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html",
                    "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "scikit-learn", "-q"], check=True)
    print("Packages installed")
else:
    print("Local mode — using conda env")

## Path configuration

In [ ]:
if COLAB_MODE:
    EDGE_FILE   = Path("dgi_edges_gnn.csv")
    FIGURES_DIR = Path("figures"); FIGURES_DIR.mkdir(exist_ok=True)
    TABLES_DIR  = Path("tables");  TABLES_DIR.mkdir(exist_ok=True)
    MODELS_DIR  = Path("models");  MODELS_DIR.mkdir(exist_ok=True)
    if not EDGE_FILE.exists():
        raise FileNotFoundError(
            "Upload dgi_edges_gnn.csv via Files → Upload in the sidebar.")
else:
    def _find_repo_root(start):
        for p in [start, *start.parents]:
            if (p / "paths.py").exists():
                return p
        raise FileNotFoundError(
            "paths.py not found — run: python scripts/data_download.py")

    REPO_ROOT = _find_repo_root(Path.cwd())
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
    if str(REPO_ROOT / "scripts") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "scripts"))

    from paths import PROC_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR
    EDGE_FILE = TABLES_DIR / "dgi_edges_gnn.csv"

print(f"Edge file : {EDGE_FILE}")
print(f"Figures   : {FIGURES_DIR}")
print(f"Models    : {MODELS_DIR}")

## Imports & configuration

In [ ]:
import torch
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from gnn_functions import (
    build_graph, make_edge_tensors,
    GCNModel, GATModel, SAGEModel, param_count,
    train_model, evaluate_model, rank_drugs, export_results,
    plot_training, plot_comparison, plot_scatter, plot_ranking,
)

# ── Hyperparameters ───────────────────────────────────────────────────────────
HIDDEN_DIM   = 128
EMBED_DIM    = 64
DROPOUT      = 0.3
LR           = 0.005
WEIGHT_DECAY = 1e-4
N_EPOCHS     = 300
PATIENCE     = 40
GAT_HEADS    = 4
RANDOM_SEED  = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {DEVICE}")

## 1 · Build drug–gene graph

In [ ]:
edges_df = pd.read_csv(EDGE_FILE)
print(f"Edges: {len(edges_df)}  |  genes: {edges_df.gene.nunique()}  |  drugs: {edges_df.drug.nunique()}")

(graph_data, node2idx, idx2node, labels,
 gene_set, drug_set, scaler,
 tr_idx, va_idx, te_idx) = build_graph(edges_df, DEVICE, RANDOM_SEED)

feat_dim = graph_data.x.shape[1]

## 2 · Define & compare models

In [ ]:
models_cfg = {
    "GCN"       : GCNModel(feat_dim, HIDDEN_DIM, EMBED_DIM, DROPOUT),
    "GAT"       : GATModel(feat_dim, HIDDEN_DIM//GAT_HEADS, EMBED_DIM, GAT_HEADS, DROPOUT),
    "GraphSAGE" : SAGEModel(feat_dim, HIDDEN_DIM, EMBED_DIM, DROPOUT),
}
for name, model in models_cfg.items():
    print(f"  {name:<12}: {param_count(model):,} parameters")

## 3 · Train all architectures

In [ ]:
all_results = {}

for name, model in models_cfg.items():
    print(f"\n[{name}]")
    history, best_val = train_model(
        model, graph_data, edges_df, node2idx, labels,
        tr_idx, va_idx, DEVICE,
        lr=LR, weight_decay=WEIGHT_DECAY,
        n_epochs=N_EPOCHS, patience=PATIENCE,
    )
    test_m = evaluate_model(
        model, graph_data, edges_df, node2idx, labels, te_idx, DEVICE)
    with torch.no_grad():
        embeds = model.encode(graph_data.x,
                              graph_data.edge_index).cpu().numpy()
    ranking = rank_drugs(model, graph_data, edges_df, node2idx, DEVICE)
    all_results[name] = {
        "history"   : history,
        "best_val"  : best_val,
        "test"      : test_m,
        "embeddings": embeds,
        "ranking"   : ranking,
        "model"     : model,
    }
    m = test_m
    print(f"  Test → R²={m['r2']:.4f}  MSE={m['mse']:.4f}  MAE={m['mae']:.4f}")

best_name  = max(all_results, key=lambda n: all_results[n]["test"]["r2"])
best_model = all_results[best_name]["model"]
print(f"\nBest model: {best_name}  R²={all_results[best_name]['test']['r2']:.4f}")

## 4 · Performance figures

In [ ]:
plot_training(all_results, best_name, FIGURES_DIR)
plot_comparison(all_results, FIGURES_DIR)
plot_scatter(all_results, best_name, FIGURES_DIR)

## 5 · Drug candidate ranking

In [ ]:
ranking = all_results[best_name]["ranking"]
plot_ranking(ranking, best_name, FIGURES_DIR, top_n=25)
print(f"\nTop 10 ({best_name}):")
ranking[["rank","drug","gene","gnn_score","approved","clinical_phase"]].head(10)

## 6 · Save model & export results

In [ ]:
export_results(
    best_name, best_model, all_results,
    edges_df, node2idx, idx2node, gene_set,
    EMBED_DIM, scaler, MODELS_DIR, TABLES_DIR,
)